# Árvores de Decisão

Dois exemplos com Python + scikit-learn:

| | Pergunta | Mede a qualidade do nó com | Escolhe a pergunta pela |
|---|---|---|---|
| **Parte 1 — Classificação** | A pessoa vai presencial? (Sim/Não) | Entropia | Ganho de Informação |
| **Parte 2 — Regressão** | Quantos litros de cerveja São Paulo consome no dia? | MSE | Redução do MSE |

Paleta em todos os gráficos: **Preto, Branco, Laranja e Azul Escuro**.

# Parte 1 — Classificação: vai presencial?

**Exemplo 1** — Árvore de Decisão com Python + scikit-learn, usando **Entropia e Ganho de Informação**.

Regras do modelo híbrido sem escala:
- **Cientista de dados** → vai presencial só na **segunda-feira**.
- **Demais pessoas da área** → vão **dois dias**: **segunda** e **terça** (dia preferencial).
- Ou seja, **segunda-feira é presencial para todos**.

Cada linha é **uma pessoa em um dia**. Variáveis (Sim = 1, Não = 0):

| variável | pergunta |
|---|---|
| `cientista_dados` | A pessoa é cientista de dados? |
| `e_segunda` | O dia é segunda-feira? |
| `e_terca` | O dia é terça-feira? |
| **`presencial`** (alvo) | **A pessoa vai presencial?** |

Paleta: **Preto, Branco, Laranja (Presencial = Sim) e Azul Escuro (Presencial = Não)**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

PRETO, BRANCO, LARANJA, AZUL_ESCURO = "#000000", "#FFFFFF", "#F28C28", "#0B2545"
COR_CLASSE = {"Sim": LARANJA, "Não": AZUL_ESCURO}

plt.rcParams.update({
    "figure.facecolor": BRANCO, "axes.facecolor": BRANCO,
    "axes.edgecolor": PRETO, "axes.labelcolor": PRETO, "text.color": PRETO,
    "xtick.color": PRETO, "ytick.color": PRETO, "font.size": 11,
})

## 1. Dados

In [ ]:
df = pd.DataFrame({
    "cientista_dados": [1, 1, 1, 0, 0, 0],
    "e_segunda":       [1, 0, 0, 1, 0, 0],
    "e_terca":         [0, 1, 0, 0, 1, 0],
})

def regra(linha):
    if linha["e_segunda"] == 1:  # segunda é presencial para todos
        return "Sim"
    if linha["e_terca"] == 1 and linha["cientista_dados"] == 0:  # demais também vão na terça
        return "Sim"
    return "Não"

df["presencial"] = df.apply(regra, axis=1)
FEATURES = ["e_segunda", "cientista_dados", "e_terca"]
df

## 2. Intuição: traçando "retas" no quadrante
Eixo X = **É cientista de dados?** · Eixo Y = **É segunda?**

Nos dias que não são segunda, o ponto de cima é a **terça** e o de baixo é **outro dia**.

A árvore procura retas (perguntas Sim/Não) que deixem cada região com uma só cor:
1. **É segunda?** → a metade de cima fica toda laranja.
2. **É cientista?** → embaixo à direita fica todo azul.
3. **É terça?** → resolve o único quadrante que ainda estava misturado.

In [ ]:
def posicao(linha):
    x = linha["cientista_dados"]
    y = linha["e_segunda"]
    if y == 0:
        y += 0.15 if linha["e_terca"] else -0.15
    return x, y


def quadrante(ax, passo, titulo):
    for _, linha in df.iterrows():
        x, y = posicao(linha)
        ax.scatter(x, y, s=380, c=COR_CLASSE[linha["presencial"]], edgecolors=PRETO, zorder=3)
        if linha["e_segunda"] == 0:
            ax.annotate("terça" if linha["e_terca"] else "outro dia", (x, y),
                        xytext=(16, -4), textcoords="offset points", fontsize=9)
    if passo >= 1:
        ax.plot([-0.5, 1.5], [0.5, 0.5], color=PRETO, lw=2, ls="--")
    if passo >= 2:
        ax.plot([0.5, 0.5], [-0.5, 0.5], color=PRETO, lw=2, ls="--")
    if passo >= 3:
        ax.plot([-0.5, 0.5], [0, 0], color=PRETO, lw=2, ls="--")
    ax.set_xticks([0, 1], ["Não", "Sim"]); ax.set_yticks([0, 1], ["Não", "Sim"])
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("É cientista de dados?"); ax.set_ylabel("É segunda?")
    ax.set_title(titulo)


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, passo, titulo in zip(axes, [1, 2, 3], ["1ª reta: É segunda?", "2ª reta: É cientista?", "3ª reta: É terça?"]):
    quadrante(ax, passo, titulo)

handles = [plt.Line2D([], [], marker="o", ls="", markersize=12, markerfacecolor=c,
                      markeredgecolor=PRETO, label=f"Presencial = {k}") for k, c in COR_CLASSE.items()]
fig.legend(handles=handles, loc="upper center", ncol=2, edgecolor=PRETO, bbox_to_anchor=(0.5, 1.06))
plt.tight_layout(); plt.show()

## 3. Entropia e Ganho de Informação
Por que a árvore pergunta **"É segunda?"** primeiro? Ela mede a **mistura** de cada grupo com a **Entropia**
e escolhe a pergunta que mais reduz essa mistura — o **Ganho de Informação**.

$$H(S) = -\sum_i p_i \log_2 p_i$$

- $H = 0$ → grupo puro (todos Sim ou todos Não)
- $H = 1$ → metade Sim, metade Não (mistura máxima)

$$IG(S, A) = H(S) - \sum_{v \in \{Sim,\,Não\}} \frac{|S_v|}{|S|}\, H(S_v)$$

### Exemplo passo a passo

**Parte A — Entropia da raiz**

1. **Contamos as classes:** 6 casos → **3 Sim** e **3 Não**.
2. **Proporções:** $p(Sim) = 3/6 = 0{,}5$ e $p(Não) = 3/6 = 0{,}5$
3. **Log₂ de cada proporção:** "2 elevado a quanto dá 0,5?" → $\log_2(0{,}5) = -1$.
   O log de um número menor que 1 é sempre negativo — por isso a fórmula tem um sinal de menos na frente.
4. **Proporção × log:** $0{,}5 \times (-1) = -0{,}5$ para cada classe.
5. **Somamos e trocamos o sinal:** $H(raiz) = -(-0{,}5 - 0{,}5) = 1$ → **mistura máxima**.

**Parte B — Ganho de "É segunda?"**

6. **Separamos os casos:** ramo Sim → 2 casos (2 Sim, 0 Não) · ramo Não → 4 casos (1 Sim, 3 Não).
7. **Entropia do ramo Sim:** grupo puro → $p(Sim) = 1$, $\log_2(1) = 0$ → $H = 0$.
8. **Entropia do ramo Não — proporções e logs:**
   $p(Sim) = 1/4 = 0{,}25 \Rightarrow \log_2(0{,}25) = -2$ · $p(Não) = 3/4 = 0{,}75 \Rightarrow \log_2(0{,}75) \approx -0{,}415$
9. **Produto e soma:** $0{,}25 \times (-2) = -0{,}500$ · $0{,}75 \times (-0{,}415) = -0{,}311$ → $H = -(-0{,}500 - 0{,}311) = 0{,}811$
10. **Média ponderada dos ramos** (2 de 6 casos no ramo Sim, 4 de 6 no ramo Não):
    $H(filhos) = \tfrac{2}{6} \times 0 + \tfrac{4}{6} \times 0{,}811 = 0{,}541$
11. **Ganho:** $IG = H(raiz) - H(filhos) = 1 - 0{,}541 = 0{,}459$

A mesma conta em Python, mostrando cada passo:

In [ ]:
from math import log2

# Parte A — raiz
sim, nao = 3, 3
total = sim + nao
p_sim, p_nao = sim / total, nao / total
print(f"1. Contagem: {sim} Sim, {nao} Não, total {total}")
print(f"2. Proporções: p(Sim) = {p_sim}, p(Não) = {p_nao}")
print(f"3. Logs: log2({p_sim}) = {log2(p_sim)}, log2({p_nao}) = {log2(p_nao)}")
print(f"4. Produtos: {p_sim * log2(p_sim)} e {p_nao * log2(p_nao)}")
h_raiz = -(p_sim * log2(p_sim) + p_nao * log2(p_nao))
print(f"5. H(raiz) = {h_raiz:.3f}")

# Parte B — ganho de "É segunda?"
print("\n6. Ramo Sim: 2 Sim / 0 Não  |  Ramo Não: 1 Sim / 3 Não")
h_ramo_sim = -(1 * log2(1))
print(f"7. H(ramo Sim) = {abs(h_ramo_sim):.3f}  (grupo puro)")
p1, p2 = 1 / 4, 3 / 4
print(f"8. p = {p1} → log2 = {log2(p1):.3f}  |  p = {p2} → log2 = {log2(p2):.3f}")
h_ramo_nao = -(p1 * log2(p1) + p2 * log2(p2))
print(f"9. {p1 * log2(p1):.3f} + ({p2 * log2(p2):.3f}) → H(ramo Não) = {h_ramo_nao:.3f}")
h_filhos = 2 / 6 * 0 + 4 / 6 * h_ramo_nao
print(f"10. H(filhos) = 2/6 × 0 + 4/6 × {h_ramo_nao:.3f} = {h_filhos:.3f}")
print(f"11. Ganho = {h_raiz:.3f} − {h_filhos:.3f} = {h_raiz - h_filhos:.3f}")

### Repetindo para todas as perguntas, nível a nível

In [ ]:
def entropia(y):
    p = y.value_counts(normalize=True)
    return abs(float(-(p * np.log2(p)).sum()))  # abs evita exibir -0.0


def tabela_ganhos(dados, features, alvo="presencial"):
    # Para cada variável: composição dos ramos, entropia e ganho
    linhas = []
    for f in features:
        sim, nao = dados[dados[f] == 1], dados[dados[f] == 0]
        h_filhos = len(sim) / len(dados) * entropia(sim[alvo]) + len(nao) / len(dados) * entropia(nao[alvo])
        linhas.append({
            "variável": f,
            "ramo Sim": f"{(sim[alvo] == 'Sim').sum()} Sim / {(sim[alvo] == 'Não').sum()} Não",
            "H ramo Sim": round(entropia(sim[alvo]), 3),
            "ramo Não": f"{(nao[alvo] == 'Sim').sum()} Sim / {(nao[alvo] == 'Não').sum()} Não",
            "H ramo Não": round(entropia(nao[alvo]), 3),
            "ganho": round(entropia(dados[alvo]) - h_filhos, 3),
        })
    return pd.DataFrame(linhas).sort_values("ganho", ascending=False)

### Nível 1 — raiz

In [ ]:
print(f"Entropia da raiz: {entropia(df['presencial']):.3f}")
tabela_ganhos(df, FEATURES)

`e_segunda` vence com folga. Repare em `e_terca`: ganho **zero** na raiz — sozinha ela não ajuda (mas vai ser decisiva depois).

### Nível 2 — ramo "não é segunda"

In [ ]:
nivel2 = df[df["e_segunda"] == 0]
print(f"Entropia do nó: {entropia(nivel2['presencial']):.3f}")
tabela_ganhos(nivel2, ["cientista_dados", "e_terca"])

**Empate!** `cientista_dados` e `e_terca` têm o mesmo ganho (0,311). Empates acontecem: qualquer uma das duas leva a uma
árvore pura no final. Escolhemos `cientista_dados` (mesma ordem do quadrante). O scikit-learn desempata de forma
aleatória, por isso fixamos o `random_state`.

### Nível 3 — ramo "não é segunda" e "não é cientista"

In [ ]:
nivel3 = nivel2[nivel2["cientista_dados"] == 0]
print(f"Entropia do nó: {entropia(nivel3['presencial']):.3f}")
tabela_ganhos(nivel3, ["e_terca"])

Ganho = 1: a pergunta "É terça?" elimina toda a mistura que restava. Todas as folhas ficam puras.

## 4. A árvore com scikit-learn
`criterion="entropy"` faz o sklearn usar exatamente esses cálculos.

In [ ]:
def desenhar_arvore(modelo, features, tamanho=(11, 6)):
    fig, ax = plt.subplots(figsize=tamanho)
    nos = plot_tree(modelo, feature_names=features, class_names=list(modelo.classes_),
                    filled=True, rounded=True, impurity=True, fontsize=11, ax=ax)

    # Aplica a paleta: folhas laranja/azul escuro, nós internos brancos
    for texto in nos:
        conteudo = texto.get_text()
        if "class = " in conteudo and "<=" not in conteudo:
            cor = COR_CLASSE[conteudo.split("class = ")[1].strip()]
            texto.set_color(BRANCO if cor == AZUL_ESCURO else PRETO)
        else:
            cor = BRANCO
            texto.set_color(PRETO)
        texto.set_bbox(dict(boxstyle="round", facecolor=cor, edgecolor=PRETO, linewidth=1.5))
        if texto.arrow_patch is not None:
            texto.arrow_patch.set_color(PRETO)
    plt.show()


X, y = df[FEATURES], df["presencial"]

# random_state fixo: desempata cientista_dados x e_terca sempre do mesmo jeito
arvore = DecisionTreeClassifier(criterion="entropy", random_state=2)
arvore.fit(X, y)

print(export_text(arvore, feature_names=FEATURES))
desenhar_arvore(arvore, FEATURES)

> `e_segunda <= 0.5` → **True** = "Não é segunda"; **False** = "É segunda".

---
## 5. Extrapolando: e se a segunda for feriado?
Nova variável `e_feriado`: se a segunda é feriado, **ninguém vai presencial**.
Acrescentamos 2 linhas — uma segunda-feriado para cientista e outra para os demais.

In [ ]:
df2 = df.copy()
df2["e_feriado"] = 0
df2 = pd.concat([df2, pd.DataFrame([
    {"cientista_dados": 1, "e_segunda": 1, "e_terca": 0, "e_feriado": 1, "presencial": "Não"},
    {"cientista_dados": 0, "e_segunda": 1, "e_terca": 0, "e_feriado": 1, "presencial": "Não"},
])], ignore_index=True)

FEATURES2 = ["e_feriado", "e_segunda", "cientista_dados", "e_terca"]
df2[FEATURES2 + ["presencial"]]

In [ ]:
print(f"Entropia da raiz: {entropia(df2['presencial']):.3f}")
tabela_ganhos(df2, FEATURES2)

**Surpresa:** a intuição diria para perguntar "É feriado?" só **dentro** do ramo da segunda.
Mas o algoritmo coloca **feriado na raiz**: é a pergunta que mais limpa a mistura de uma vez.

A Árvore de Decisão é **gulosa**: escolhe a melhor pergunta **em cada passo**, não a que parece mais lógica para nós.

No ramo "não é feriado" sobram exatamente os 6 casos do exemplo base — então ali dentro cresce a mesma árvore de antes.

In [ ]:
arvore2 = DecisionTreeClassifier(criterion="entropy", random_state=2)
arvore2.fit(df2[FEATURES2], df2["presencial"])

print(export_text(arvore2, feature_names=FEATURES2))
desenhar_arvore(arvore2, FEATURES2, tamanho=(13, 8))

## 6. Classificando um novo dado
Preencha as respostas e veja o caminho que o dado percorre na árvore final (com feriado).

In [ ]:
def classificar(modelo, features, novo):
    t = modelo.tree_
    no = 0
    while t.children_left[no] != -1:
        f = features[t.feature[no]]
        valor = novo[f]
        print(f"{f}? {'Sim' if valor else 'Não'}")
        no = t.children_right[no] if valor > t.threshold[no] else t.children_left[no]
    print("Presencial?", modelo.predict(pd.DataFrame([novo])[features])[0])


# Altere os valores (1 = Sim, 0 = Não) e rode a célula de novo
novo = {"e_feriado": 0, "cientista_dados": 0, "e_segunda": 0, "e_terca": 1}
classificar(arvore2, FEATURES2, novo)

## 7. Pontos positivos e negativos da Árvore de Decisão (classificação)

**Pontos positivos**
- **Fácil de explicar:** o modelo vira regras Sim/Não que qualquer pessoa lê — "se é segunda, vai presencial".
- **Captura interações sozinha:** "terça" só importa para quem não é cientista; a árvore descobriu isso sem a gente dizer.
- **Seleciona variáveis:** pergunta que não ajuda não entra (terça teve ganho 0 na raiz).
- **Pouco preparo dos dados:** não precisa normalizar nem padronizar; aceita variáveis Sim/Não, categóricas e numéricas.
- **Rápida** para treinar e para prever.

**Pontos negativos**
- **Gulosa:** escolhe a melhor pergunta em cada passo, não a melhor árvore — vimos "feriado" ir para a raiz.
- **Instável:** 2 linhas novas (os feriados) mudaram a raiz inteira; empates dependem do `random_state`.
- **Overfitting:** deixada crescer até as folhas ficarem puras, ela decora os dados. Em dados reais, com ruído,
  é preciso limitar (`max_depth`, `min_samples_leaf`) ou podar.
- **Só cortes retos:** as divisões são sempre paralelas aos eixos (as "retas" do quadrante); uma fronteira diagonal vira uma escada.
- **Viés da entropia:** variáveis com muitos valores distintos (um ID, por exemplo) parecem ter ganho alto sem ter valor real.

---

# Parte 2 — Regressão: consumo de cerveja em São Paulo

**Exemplo 2** — Árvore de Regressão com Python + scikit-learn.

No exemplo 1 a árvore respondia **"qual classe?"** (Presencial Sim/Não). Agora ela responde **"quanto?"**:
quantos litros de cerveja serão consumidos em um dia.

Dataset: *Beer Consumption – São Paulo* (Kaggle) — consumo diário de cerveja em 2015, junto com o clima do dia.

| variável | significado |
|---|---|
| `temp_max` | Temperatura máxima do dia (°C) |
| `fim_de_semana` | O dia é sábado ou domingo? (1 = Sim, 0 = Não) |
| `chuva_mm` | Precipitação do dia (mm) — entra só na parte 5 |
| **`consumo_litros`** (alvo) | **Consumo de cerveja no dia (litros)** |

> No CSV o consumo aparece como `25.461`: o ponto é **separador de milhar**, ou seja, **25.461 litros** (cerca de 25 mil litros por dia).

Paleta: **Preto, Branco, Laranja (fim de semana) e Azul Escuro (dia útil)**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text

PRETO, BRANCO, LARANJA, AZUL_ESCURO = "#000000", "#FFFFFF", "#F28C28", "#0B2545"
COR_DIA = {1: LARANJA, 0: AZUL_ESCURO}
NOME_DIA = {1: "Fim de semana", 0: "Dia útil"}

# formatação brasileira: 25401.4 -> "25.401"
num = lambda x: f"{x:,.0f}".replace(",", ".")
litros = lambda x: num(x) + " L"

plt.rcParams.update({
    "figure.facecolor": BRANCO, "axes.facecolor": BRANCO,
    "axes.edgecolor": PRETO, "axes.labelcolor": PRETO, "text.color": PRETO,
    "xtick.color": PRETO, "ytick.color": PRETO, "font.size": 11,
})

## 1. Dados
O CSV usa **vírgula decimal** nas temperaturas e **ponto como separador de milhar** no consumo (`25.461` = 25.461 litros).
Também traz linhas vazias no fim. Limpamos e ficamos com os 365 dias de 2015.

In [ ]:
bruto = pd.read_csv("Consumo_cerveja.csv", decimal=",", thousands=".").dropna(how="all")
df_cerveja = bruto.rename(columns={
    "Temperatura Maxima (C)": "temp_max",
    "Final de Semana": "fim_de_semana",
    "Precipitacao (mm)": "chuva_mm",
    "Consumo de cerveja (litros)": "consumo_litros",
})[["Data", "temp_max", "fim_de_semana", "chuva_mm", "consumo_litros"]]
df_cerveja["fim_de_semana"] = df_cerveja["fim_de_semana"].astype(int)
df_cerveja["consumo_litros"] = df_cerveja["consumo_litros"].astype(float)

FEATURES_REG = ["temp_max", "fim_de_semana"]
ALVO = "consumo_litros"
print(len(df_cerveja), "dias")
df_cerveja.head()

In [ ]:
df_cerveja[["temp_max", "chuva_mm", "consumo_litros"]].describe().round(1)

**Sem árvore nenhuma**, o melhor palpite para qualquer dia é a **média** do consumo.
Medimos o erro desse palpite com o **MSE** (erro quadrático médio) e o **MAE** (erro absoluto médio, em litros).

In [ ]:
media = df_cerveja[ALVO].mean()
print(f"Média geral: {litros(media)}")
print(f"MSE prevendo sempre a média: {num(((df_cerveja[ALVO] - media) ** 2).mean())} (litros²)")
print(f"MAE prevendo sempre a média: {litros((df_cerveja[ALVO] - media).abs().mean())}")

## 2. Intuição: cortes e "degraus"
Na classificação, as retas separavam **cores**. Na regressão, cada corte divide os dias em grupos e a previsão de cada grupo é
a **média** do consumo ali dentro — uma linha horizontal. O resultado é uma previsão em **degraus**.

1. **Sem pergunta:** uma linha só, na média geral.
2. **1º corte — temperatura máxima ≤ 25,85 °C?** Dias frios de um lado, quentes do outro, cada um com sua média.
3. **2º corte — é fim de semana?** Dentro de cada lado, fim de semana (laranja) bebe mais que dia útil (azul).

In [ ]:
LIMIAR = 25.85
x_min, x_max = df_cerveja["temp_max"].min() - 0.5, df_cerveja["temp_max"].max() + 0.5


def dispersao(ax, passo, titulo):
    for fds, grupo in df_cerveja.groupby("fim_de_semana"):
        ax.scatter(grupo["temp_max"], grupo[ALVO], s=22, c=COR_DIA[fds], edgecolors=PRETO,
                   linewidths=0.3, alpha=0.85, label=NOME_DIA[fds])
    if passo == 0:
        ax.hlines(df_cerveja[ALVO].mean(), x_min, x_max, color=PRETO, lw=3)
    if passo >= 1:
        ax.axvline(LIMIAR, color=PRETO, lw=2, ls="--")
    for lado, (a, b) in enumerate([(x_min, LIMIAR), (LIMIAR, x_max)]):
        grupo = df_cerveja[(df_cerveja["temp_max"] > LIMIAR) == bool(lado)]
        if passo == 1:
            ax.hlines(grupo[ALVO].mean(), a, b, color=PRETO, lw=3)
        if passo == 2:
            for fds in (0, 1):
                ax.hlines(grupo[grupo["fim_de_semana"] == fds][ALVO].mean(), a, b, color=COR_DIA[fds], lw=4)
    ax.set_xlim(x_min, x_max)
    ax.set_xlabel("Temperatura máxima (°C)"); ax.set_ylabel("Consumo (litros)")
    ax.set_title(titulo)


fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, passo, titulo in zip(axes, [0, 1, 2], ["Sem pergunta: média geral", "1º corte: temp. máx ≤ 25,85?", "2º corte: fim de semana?"]):
    dispersao(ax, passo, titulo)
axes[0].legend(edgecolor=PRETO)
plt.tight_layout(); plt.show()

## 3. MSE e Redução do erro
Na classificação medíamos a **mistura** com a Entropia. Na regressão medimos o **espalhamento** dos valores em torno da média
com o **MSE** (é a variância do grupo):

$$MSE(S) = \frac{1}{n}\sum_{i=1}^{n} (y_i - \bar{y})^2$$

- $MSE = 0$ → todos os dias do grupo têm o mesmo consumo (grupo "puro")
- quanto maior, mais espalhado o consumo e pior a média como previsão
- a unidade é **litros²**, por isso os números são grandes; $\sqrt{MSE}$ volta para litros

O papel do Ganho de Informação passa para a **Redução do MSE**:

$$\Delta MSE = MSE(pai) - \sum_{filhos} \frac{n_{filho}}{n_{pai}}\, MSE(filho)$$

Diferença importante: `temp_max` é **numérica**, então a árvore precisa escolher **onde cortar**. Ela testa todos os
limiares possíveis (o ponto médio entre dois valores vizinhos) e fica com o de maior redução.

### Exemplo passo a passo na raiz

1. **Raiz:** 365 dias, média **25.401 L**, MSE **19.299.436** (≈ 19,3 milhões de litros²).
2. **Pergunta "É fim de semana?":** dia útil → 261 dias, média 23.998 L, MSE 14.290.394 · fim de semana → 104 dias, média 28.923 L, MSE 14.529.301.
3. **MSE ponderado dos filhos:** $\tfrac{261}{365} \times 14.290.394 + \tfrac{104}{365} \times 14.529.301 = 14.358.466$
4. **Redução:** $19.299.436 - 14.358.466 = 4.940.970$
5. **Pergunta "temp_max ≤ limiar?":** testamos todos os limiares. O melhor é **25,85 °C**:
   frio → 143 dias, média 22.303 L, MSE 9.825.960 · quente → 222 dias, média 27.397 L, MSE 15.235.459.
6. **MSE ponderado:** $\tfrac{143}{365} \times 9.825.960 + \tfrac{222}{365} \times 15.235.459 = 13.116.121$
7. **Redução:** $19.299.436 - 13.116.121 = 6.183.315$ → maior que 4.940.970 → **temperatura vira a raiz**.

A mesma conta em Python:

In [ ]:
def mse(y):
    return float(((y - y.mean()) ** 2).mean()) if len(y) else 0.0


def reducao(dados, feature, limiar):
    esq = dados[dados[feature] <= limiar][ALVO]
    dir_ = dados[dados[feature] > limiar][ALVO]
    filhos = (len(esq) * mse(esq) + len(dir_) * mse(dir_)) / len(dados)
    return mse(dados[ALVO]) - filhos, esq, dir_


pai = df_cerveja[ALVO]
print(f"1. Raiz: {len(pai)} dias, média {litros(pai.mean())}, MSE {num(mse(pai))}")
for passo, (f, limiar, nomes) in enumerate([("fim_de_semana", 0.5, ("dia útil", "fim de semana")),
                                             ("temp_max", LIMIAR, ("frio", "quente"))]):
    red, esq, dir_ = reducao(df_cerveja, f, limiar)
    print(f"\n{f} (limiar {limiar}):")
    for nome, g in zip(nomes, (esq, dir_)):
        print(f"   {nome:>13}: {len(g)} dias, média {litros(g.mean())}, MSE {num(mse(g))}")
    filhos = mse(pai) - red
    print(f"   MSE ponderado = {len(esq)}/{len(pai)} × {num(mse(esq))} + {len(dir_)}/{len(pai)} × {num(mse(dir_))} = {num(filhos)}")
    print(f"   Redução = {num(mse(pai))} − {num(filhos)} = {num(red)}")

### Como a árvore escolhe o limiar da temperatura
Ela calcula a redução para **cada** limiar candidato. O pico da curva é o corte escolhido.

In [ ]:
def melhor_corte(dados, feature):
    valores = np.sort(dados[feature].unique())
    limiares = (valores[:-1] + valores[1:]) / 2
    reducoes = np.array([reducao(dados, feature, t)[0] for t in limiares])
    return limiares, reducoes


limiares, reducoes = melhor_corte(df_cerveja, "temp_max")
melhor = limiares[reducoes.argmax()]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(limiares, reducoes, color=AZUL_ESCURO, lw=2)
ax.axvline(melhor, color=LARANJA, lw=3, ls="--")
ax.scatter([melhor], [reducoes.max()], s=120, c=LARANJA, edgecolors=PRETO, zorder=3)
ax.annotate(f"melhor limiar = {melhor:.2f} °C\nredução = {num(reducoes.max())}", (melhor, reducoes.max()),
            xytext=(12, -36), textcoords="offset points")
ax.set_xlabel("Limiar testado para temp_max (°C)"); ax.set_ylabel("Redução do MSE (litros²)")
ax.set_title("Redução do MSE para cada limiar de temperatura")
plt.tight_layout(); plt.show()

### Repetindo para todas as variáveis, nível a nível

In [ ]:
def tabela_reducoes(dados, features):
    # Para cada variável: melhor limiar, composição dos dois lados e redução do MSE
    linhas = []
    for f in features:
        limiares, reducoes = melhor_corte(dados, f)
        if not len(limiares):
            continue
        t = limiares[reducoes.argmax()]
        red, esq, dir_ = reducao(dados, f, t)
        linhas.append({
            "variável": f, "limiar": round(t, 2),
            "≤ limiar": f"{len(esq)} dias · média {litros(esq.mean())}",
            "> limiar": f"{len(dir_)} dias · média {litros(dir_.mean())}",
            "redução MSE": num(red),
        })
    return pd.DataFrame(linhas).sort_values("redução MSE", key=lambda c: c.str.replace(".", "").astype(int), ascending=False)

### Nível 1 — raiz

In [ ]:
print(f"MSE da raiz: {num(mse(df_cerveja[ALVO]))}")
tabela_reducoes(df_cerveja, FEATURES_REG)

### Nível 2 — dentro de cada lado da temperatura

In [ ]:
frio, quente = df_cerveja[df_cerveja["temp_max"] <= LIMIAR], df_cerveja[df_cerveja["temp_max"] > LIMIAR]
print(f"Lado frio (≤ {LIMIAR} °C): {len(frio)} dias, MSE {num(mse(frio[ALVO]))}")
display(tabela_reducoes(frio, FEATURES_REG))
print(f"Lado quente (> {LIMIAR} °C): {len(quente)} dias, MSE {num(mse(quente[ALVO]))}")
display(tabela_reducoes(quente, FEATURES_REG))

Nos dois lados, **fim de semana** vence. A árvore para aqui (`max_depth=2`): são 4 folhas, cada uma prevendo a média do seu grupo.

## 4. A árvore com scikit-learn
`criterion="squared_error"` faz o sklearn usar exatamente a redução do MSE que calculamos.

In [ ]:
def desenhar_arvore_reg(modelo, features, tamanho=(13, 6)):
    fig, ax = plt.subplots(figsize=tamanho)
    nos = plot_tree(modelo, feature_names=features, filled=True, rounded=True, impurity=True,
                    precision=0, fontsize=11, ax=ax)

    # Aplica a paleta: folhas azul escuro com texto branco, nós internos brancos
    for texto in nos:
        folha = "value" in texto.get_text() and "<=" not in texto.get_text()
        texto.set_color(BRANCO if folha else PRETO)
        texto.set_bbox(dict(boxstyle="round", facecolor=AZUL_ESCURO if folha else BRANCO,
                            edgecolor=PRETO, linewidth=1.5))
        if texto.arrow_patch is not None:
            texto.arrow_patch.set_color(PRETO)
    plt.show()


arvore_reg = DecisionTreeRegressor(criterion="squared_error", max_depth=2, random_state=0)
arvore_reg.fit(df_cerveja[FEATURES_REG], df_cerveja[ALVO])

print(export_text(arvore_reg, feature_names=FEATURES_REG, decimals=2))  # value = litros
print(f"Corte da raiz no sklearn: {arvore_reg.tree_.threshold[0]:.2f} °C  (manual: {melhor:.2f} °C)")
desenhar_arvore_reg(arvore_reg, FEATURES_REG)

> Cada caixa mostra o `squared_error` (MSE do nó), `samples` (dias) e `value` (**a média = a previsão**).
> `True` = condição verdadeira (ramo da esquerda).

In [ ]:
def erros(modelo, features):
    previsto = modelo.predict(df_cerveja[features])
    return {"MAE": litros(np.abs(previsto - df_cerveja[ALVO]).mean()),
            "RMSE": litros(np.sqrt(((previsto - df_cerveja[ALVO]) ** 2).mean()))}


sem_arvore = {"MAE": litros((df_cerveja[ALVO] - media).abs().mean()),
              "RMSE": litros(np.sqrt(((df_cerveja[ALVO] - media) ** 2).mean()))}
pd.DataFrame({"só a média": sem_arvore, "árvore (2 níveis)": erros(arvore_reg, FEATURES_REG)}).T

---
## 5. Extrapolando: e se a chuva entrar?
Faz sentido pensar que dia chuvoso tem menos consumo. Colocamos `chuva_mm` na disputa e repetimos as contas.

In [ ]:
FEATURES_REG2 = ["temp_max", "fim_de_semana", "chuva_mm"]
print(f"MSE da raiz: {num(mse(df_cerveja[ALVO]))}")
tabela_reducoes(df_cerveja, FEATURES_REG2)

In [ ]:
print(f"Lado frio: MSE {num(mse(frio[ALVO]))}")
display(tabela_reducoes(frio, FEATURES_REG2))
print(f"Lado quente: MSE {num(mse(quente[ALVO]))}")
display(tabela_reducoes(quente, FEATURES_REG2))

**Surpresa:** a chuva **reduz** o erro (dias com mais de 2,1 mm consomem em média cerca de 2 mil litros a menos), mas
**perde em todos os nós** para temperatura e fim de semana. A árvore é **gulosa**: em cada nó fica só com a melhor pergunta.
Resultado: a árvore com chuva é **idêntica** à anterior — e o erro também.

In [ ]:
arvore_reg2 = DecisionTreeRegressor(criterion="squared_error", max_depth=2, random_state=0)
arvore_reg2.fit(df_cerveja[FEATURES_REG2], df_cerveja[ALVO])

print(export_text(arvore_reg2, feature_names=FEATURES_REG2, decimals=2))
print("Importância de cada variável:", dict(zip(FEATURES_REG2, arvore_reg2.feature_importances_.round(3))))
pd.DataFrame({"sem chuva": erros(arvore_reg, FEATURES_REG), "com chuva": erros(arvore_reg2, FEATURES_REG2)}).T

## 6. Estimando o consumo de um novo dia
Preencha o dia e veja o caminho percorrido na árvore.

In [ ]:
def prever(modelo, features, novo_dia):
    t = modelo.tree_
    no = 0
    while t.children_left[no] != -1:
        f = features[t.feature[no]]
        sim = novo_dia[f] <= t.threshold[no]
        print(f"{f} <= {t.threshold[no]:.2f}? {'Sim' if sim else 'Não'}")
        no = t.children_left[no] if sim else t.children_right[no]
    print(f"Consumo estimado: {litros(modelo.predict(pd.DataFrame([novo_dia])[features])[0])}")


# Altere os valores e rode a célula de novo
novo_dia = {"temp_max": 31.0, "fim_de_semana": 1}
prever(arvore_reg, FEATURES_REG, novo_dia)

## 7. Pontos positivos e negativos da Árvore de Regressão

**Pontos positivos**
- **Fácil de explicar:** cada folha é a média de dias parecidos — "dia quente de fim de semana ≈ 30.817 litros".
- **Captura não linearidade e interação** sem criar variáveis novas: o efeito do fim de semana aparece dentro de cada faixa de temperatura.
- **Ignora variáveis fracas:** a chuva ficou com importância 0.
- **Pouco preparo dos dados:** não precisa normalizar; temperatura (°C) e Sim/Não convivem na mesma árvore.
- **Ganho rápido:** com só 2 perguntas, o erro médio cai de 3.609 L para 2.332 L.

**Pontos negativos**
- **Previsão em degraus:** só existem 4 respostas possíveis. Um dia útil de 25,8 °C e outro de 25,9 °C recebem
  previsões que diferem em mais de 4.700 litros.
- **Não extrapola:** um dia de 40 °C recebe a mesma previsão de um dia de 26 °C (a média da folha). A árvore nunca prevê
  além do que viu no treino.
- **Overfitting:** com mais níveis surgem folhas com poucos dias — com profundidade 3 já aparece uma folha com só 6 dias.
- **Instável:** o limiar de 25,85 °C pode mudar bastante com dados de outro ano.
- **Erro ainda alto** (± 2.332 L): combinações de árvores (Random Forest, Gradient Boosting) costumam errar menos.